# 半监督 GNN 空间转录组细胞类型注释 — 卵巢癌

转录本级别流水线（不依赖细胞分割）

## 数据集
- **scRNA-seq**: 17K Ovarian Cancer Flex
- **空间转录组**: Xenium V1 FFPE Ovarian Cancer

## 执行流程

本 notebook 设计为**从头到尾顺序执行**。每个 stage 完成后会自动持久化到磁盘，
重跑时通过 `FORCE` 字典控制哪些步骤需要重做。

| Stage | 内容 | 典型耗时 | 输出 |
|---|---|---|---|
| 0 | 环境配置 + 依赖检查 | <1 min | — |
| 1 | scRNA 加载 + 类合并 | 1 min | `data/cache/export/` |
| 2 | 转录本 binning + PCA + 建图 | 30–60 min | `data/cache/graph/` |
| 3 | GNN 训练（GCN / GraphSAGE / GAT） | 30 min × 3 | `results/models/` |
| 4 | GNN 推断 + TopACT 基线 | 5 min | `results/predictions/` |
| 5 | 塌缩诊断 + P0 修复验证 | <1 min | — |
| 6 | 保存预测 CSV + 空间可视化 | 2 min | `figures/spot_*.png` |
| 7 | Moran's I（所有方法） | 5 min | `results/evaluation/morans_*.pkl` |
| 8 | Seurat 对比（需先跑 labelTransfer_ovarian.ipynb） | 2 min | `figures/spatial_comparison_*.png` |
| 9 | 消融实验（5 配置） | 15 min × 5 | `results/evaluation/ablation_*.pkl` |
| 10 | 表 + 图（论文最终交付物） | 5 min | `results/evaluation/thesis_tables.md` + 多张图 |

## 关键开关（在 Stage 0 中修改）

- `FORCE['graph']`: True 时重建 spot 图（图改 PCA/k 后必须）
- `FORCE['train']`: True 时重训 GNN
- `FORCE['eval']`: True 时重新推断
- `RUN_ABLATION`: True 时跑 Stage 9（耗时 1+ 小时）


## Stage 0 — 环境配置

设置路径、超参、强制重跑开关，选择 GPU。所有后续 cell 依赖这里定义的全局变量。


In [ ]:
# ============================================================
# Stage 0 — 环境配置
# ============================================================
import os, sys, warnings, pickle, json
import numpy as np
import pandas as pd
import torch
warnings.filterwarnings('ignore')
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

# ── 路径配置（卵巢癌数据集）──────────────────────────────────
PATHS = {
    'raw': {
        'transcripts': 'data/raw/xenium/Xenium_Prime_Ovarian_Cancer_FFPE_XRrun_outs/transcripts.parquet',
        'cells_csv'  : 'data/raw/xenium/Xenium_Prime_Ovarian_Cancer_FFPE_XRrun_outs/cells.csv.gz',
        'flex_h5'    : 'data/raw/flex/17k_Ovarian_Cancer_scFFPE_count_filtered_feature_bc_matrix.h5',
        'flex_annot' : 'data/raw/flex/FLEX_Ovarian_Barcode_Cluster_Annotation.csv',
    },
    'cache': {
        'export': 'data/cache/export/',
        'graph' : 'data/cache/graph/',
    },
    'output': {
        'models'     : 'results/models/',
        'predictions': 'results/predictions/',
        'evaluation' : 'results/evaluation/',
    },
    'figures': 'figures/',
}
for d in [*PATHS['output'].values(),
          PATHS['cache']['export'], PATHS['cache']['graph'],
          PATHS['figures']]:
    os.makedirs(d, exist_ok=True)

# ── 模型/训练超参 ─────────────────────────────────────────────
PARAMS = {
    # 数据
    'bin_size'             : 15,
    'qv_threshold'         : 20,
    'min_transcripts'      : 1,
    # 图构建
    'k_feat'               : 15,
    'k_spatial'            : 10,
    'k_cross'              : 20,        # 跨域 MNN 的 k
    # PCA 对齐
    'n_pca'                : 100,
    'pca_clip_pct'         : 99.0,
    # 类权重
    'max_weight_multiplier': 5.0,
    # 训练
    'val_ratio'            : 0.2,
    'hidden_dim'           : 128,
    'gat_heads'            : 2,
    'proj_dim'             : None,
    'dropout'              : 0.5,
    'n_epochs'             : 500,
    'lr'                   : 1e-3,
    'weight_decay'         : 5e-4,
    'patience'             : 40,
    'lr_factor'            : 0.5,
    'lr_patience'          : 15,
    'min_lr'               : 1e-5,
    'max_grad_norm'        : 1.0,
    # 域适应
    'warmup_epochs'        : 30,
    'lambda_mmd'           : 0.1,
    'lambda_ent'           : 0.01,
    'lambda_pl'            : 0.3,
    'pl_threshold'         : 0.95,
    'pl_update_freq'       : 10,
    # 系统
    'device'               : 'cpu',     # 由 select_gpu 覆写
    'seed'                 : 42,
}

# ── 强制重跑开关（True = 忽略缓存重新计算）────────────────────
FORCE = {
    'scrna_export': False,
    'graph'       : False,
    'train'       : False,
    'eval'        : False,
    'seurat_cmp'  : False,
    'ablation'    : False,    # 单独消融配置的强制重跑
}

# ── 大开关：是否运行耗时的消融实验（Stage 9）─────────────────
# 首次跑通整个流程时设 False（跳过 Stage 9）；
# 主流程稳定后再设 True 跑消融。
RUN_ABLATION = True

# ── 随机种子 ──────────────────────────────────────────────────
from utils import set_seed
set_seed(PARAMS['seed'])

# ── GPU 选择 ──────────────────────────────────────────────────
from gpu_utils import list_gpus, select_gpu
list_gpus(show_processes=True)
PARAMS['device'] = select_gpu('auto', min_free_gb=8.0)

# ── 缓存文件命名 tag（参数变化时自动隔离）─────────────────────
cache_tag = (
    f"kf{PARAMS['k_feat']}_ks{PARAMS['k_spatial']}"
    f"_kc{PARAMS['k_cross']}_pca{PARAMS['n_pca']}"
    f"_bin{PARAMS['bin_size']}_val{PARAMS['val_ratio']}"
)
weight_tag = (f"h{PARAMS['hidden_dim']}"
              f"_kf{PARAMS['k_feat']}_ks{PARAMS['k_spatial']}"
              f"_bin{PARAMS['bin_size']}")

# ── 缓存文件路径汇总 ──────────────────────────────────────────
GRAPH_FILE              = os.path.join(PATHS['cache']['graph'], f'graph_{cache_tag}.pt')
SCALER_FILE             = os.path.join(PATHS['cache']['graph'], f'scaler_{cache_tag}.pkl')
COORDS_FILE             = os.path.join(PATHS['cache']['graph'], f'spot_coords_{cache_tag}.npy')
PCA_FILE                = os.path.join(PATHS['cache']['graph'], f'pca_{cache_tag}.pkl')
TX_SAMPLE_FILE          = os.path.join(PATHS['cache']['graph'], f'transcripts_sample_{cache_tag}.npz')
RESULTS_SUMMARY_FILE    = os.path.join(PATHS['output']['models'], f'gnn_results_summary_{weight_tag}.pkl')
SPOT_PROBAS_FILE        = os.path.join(PATHS['output']['predictions'], 'spot_probas_all.pkl')
TOPACT_PROBA_FILE       = os.path.join(PATHS['output']['predictions'], 'topact_spot_proba.pkl')
TOPACT_METRIC_FILE      = os.path.join(PATHS['output']['evaluation'],  'topact_metrics.pkl')
MORAN_FILE              = os.path.join(PATHS['output']['evaluation'],  'morans_all_methods.pkl')
ABL_RESULTS_FILE        = os.path.join(PATHS['output']['evaluation'],  'ablation_results.pkl')
ABL_XENIUM_FILE         = os.path.join(PATHS['output']['evaluation'],  'ablation_xenium_metrics.pkl')
EXPORT_FILES = {
    'flex_expr'         : os.path.join(PATHS['cache']['export'], 'flex_expr.npy'),
    'flex_labels'       : os.path.join(PATHS['cache']['export'], 'flex_labels.npy'),
    'cell_types'        : os.path.join(PATHS['cache']['export'], 'cell_types.json'),
    'xenium_panel_genes': os.path.join(PATHS['cache']['export'], 'xenium_panel_genes.json'),
}

# ── 状态报告 ──────────────────────────────────────────────────
print('\n=== 原始数据检查 ===')
for name, path in PATHS['raw'].items():
    exists = os.path.exists(path)
    size   = f"{os.path.getsize(path)/1e6:.1f} MB" if exists else "不存在"
    print(f'  {name:<14} {"✅" if exists else "❌"}  {size}')

print('\n=== 缓存状态 ===')
checks = [
    ('scRNA numpy 导出', all(os.path.exists(f) for f in EXPORT_FILES.values())),
    ('spot 图',          all(os.path.exists(p) for p in [GRAPH_FILE, SCALER_FILE, COORDS_FILE, PCA_FILE])),
    ('训练权重 (GCN)',   os.path.exists(os.path.join(PATHS['output']['models'], f'GCN_{weight_tag}.pt'))),
    ('训练权重 (SAGE)',  os.path.exists(os.path.join(PATHS['output']['models'], f'GraphSAGE_{weight_tag}.pt'))),
    ('训练权重 (GAT)',   os.path.exists(os.path.join(PATHS['output']['models'], f'GAT_{weight_tag}.pt'))),
    ('预测概率',         os.path.exists(SPOT_PROBAS_FILE)),
    ('TopACT 结果',      os.path.exists(TOPACT_METRIC_FILE)),
    ('消融实验',         os.path.exists(ABL_RESULTS_FILE)),
    ("Moran's I",        os.path.exists(MORAN_FILE)),
]
for name, hit in checks:
    print(f'  {"[HIT]" if hit else "[MISS]"} {name}')

print(f'\n训练设备     : {PARAMS["device"]}')
print(f'cache_tag    : {cache_tag}')
print(f'weight_tag   : {weight_tag}')
print(f'RUN_ABLATION : {RUN_ABLATION}')


In [ ]:
# ── 依赖检查 ──────────────────────────────────────────────────
import subprocess, sys

def _ensure(pkg, import_name=None):
    try:
        __import__(import_name or pkg.split('>=')[0].split('==')[0])
    except ImportError:
        print(f'安装 {pkg} ...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '--quiet'])

_ensure('pyarrow>=14.0', 'pyarrow')
_ensure('umap-learn>=0.5', 'umap')
_ensure('scanpy>=1.9', 'scanpy')

print('✅ 依赖检查完成')


## Stage 1 — scRNA-seq 加载 + 类合并

加载 Flex scRNA-seq 数据，对齐到 Xenium gene panel；
然后把因 panel 限制不可分的肿瘤亚型合并为单一类别（解决塌缩问题的根本修复）。


In [ ]:
# ── 依赖检查：pyarrow（读 parquet 必须）────────────────────────────
try:
    import pyarrow
except ImportError:
    import subprocess, sys
    print("pyarrow 未安装，正在安装...")
    subprocess.check_call([sys.executable, "-m", "pip", "install",
                           "pyarrow>=14.0", "--quiet"])
    print("pyarrow 安装完成，继续...")

# ============================================================
# Cell 2: 加载 Flex scRNA-seq 数据
#
# 卵巢版：scanpy 直接加载 .h5 + CSV 标注（无 R 依赖）
#
# Xenium panel gene 来源：从 transcripts 文件的 gene 列提取
# ============================================================
import scanpy as sc

EXPORT_OK = (all(os.path.exists(f) for f in EXPORT_FILES.values())
             and not FORCE['scrna_export'])

if EXPORT_OK:
    print('[Cache HIT] 从 numpy 缓存加载 scRNA 数据...')
    flex_expr = np.load(EXPORT_FILES['flex_expr'])
    flex_labels = np.load(EXPORT_FILES['flex_labels'])
    with open(EXPORT_FILES['cell_types']) as f:
        cell_types_list = json.load(f)
    with open(EXPORT_FILES['xenium_panel_genes']) as f:
        xenium_panel_genes = json.load(f)

else:
    print('[Cache MISS] 从 .h5 和 CSV 加载 Flex scRNA-seq...')

    # ── 1. 加载 .h5 计数矩阵 ─────────────────────────────────
    adata = sc.read_10x_h5(PATHS['raw']['flex_h5'])
    adata.var_names_make_unique()
    print(f'  原始矩阵: {adata.n_obs} 细胞 × {adata.n_vars} 基因')

    # ── 2. 加载预标注 CSV ────────────────────────────────────
    annot = pd.read_csv(PATHS['raw']['flex_annot'])
    print(f'  CSV 列名: {list(annot.columns)}')
    print(annot.head(3).to_string())

    # 自动识别 barcode 列 和 cell_type 列
    bc_col = next((c for c in annot.columns
                   if any(k in c.lower() for k in
                          ['barcode', 'cell_id', 'cellid'])), annot.columns[0])
    ct_col = next((c for c in annot.columns
                   if any(k in c.lower() for k in
                          ['cell_type', 'celltype', 'annotation',
                           'cluster_name', 'label', 'type'])), None)
    if ct_col is None:
        raise ValueError(f'找不到 cell_type 列，请手动指定。列名: {list(annot.columns)}')

    print(f'  barcode 列: {bc_col}')
    print(f'  cell_type 列: {ct_col}')

    annot = annot.set_index(bc_col)
    common_bc = adata.obs_names.intersection(annot.index)
    print(f'  有标注细胞: {len(common_bc):,} / {adata.n_obs:,}')
    if len(common_bc) == 0:
        raise ValueError('barcode 无交集！请检查 CSV barcode 格式是否与 .h5 一致')

    adata = adata[common_bc].copy()
    adata.obs['cell_type'] = annot.loc[common_bc, ct_col].values

    # ── 3. 基础 QC ────────────────────────────────────────────
    sc.pp.filter_cells(adata, min_genes=200)
    sc.pp.filter_genes(adata, min_cells=3)
    adata.var['mt'] = adata.var_names.str.startswith('MT-')
    sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], inplace=True)
    adata = adata[adata.obs['pct_counts_mt'] < 25].copy()
    print(f'  QC 后: {adata.n_obs:,} 细胞 × {adata.n_vars:,} 基因')

    # ── 4. 过滤：只保留有明确 cell_type 的细胞 ────────────────
    valid_mask = adata.obs['cell_type'].notna() & (adata.obs['cell_type'] != '')
    adata = adata[valid_mask].copy()
    print(f'  有标注细胞: {adata.n_obs:,}')
    print('\n  细胞类型分布:')
    print(adata.obs['cell_type'].value_counts().to_string())

    # ── 5. 提取 Xenium gene panel ─────────────────────────────
    #    从 transcripts 文件的 gene 列提取，比 gene_panel.json 更可靠
    print('\n  提取 Xenium gene panel...')
    tx_path = PATHS['raw']['transcripts']
    if tx_path.endswith('.parquet'):
        tx_sample = pd.read_parquet(tx_path, columns=['feature_name'])
        xen_genes = tx_sample['feature_name'].dropna().unique().tolist()
    else:
        tx_sample = pd.read_csv(tx_path, usecols=lambda c: 'gene' in c.lower()
                                or 'feature' in c.lower(), nrows=None)
        gene_col  = [c for c in tx_sample.columns
                     if 'gene' in c.lower() or 'feature' in c.lower()][0]
        xen_genes = tx_sample[gene_col].dropna().unique().tolist()

    # 去掉 Xenium 控制探针（BLANK_ / NegCtrl / Unassigned）
    xen_genes = [g for g in xen_genes
                 if not any(p in g for p in
                            ['BLANK', 'NegCtrl', 'Unassigned', 'Negative'])]
    print(f'  Xenium 基因数: {len(xen_genes)}')

    # ── 6. 特征对齐（取共同基因）─────────────────────────────
    common_genes = sorted(set(adata.var_names) & set(xen_genes))
    print(f'  共同基因数: {len(common_genes)}')
    if len(common_genes) < 50:
        raise ValueError(f'共同基因 {len(common_genes)} 个，过少！请检查基因命名格式')

    # ── 7. 提取原始 counts 矩阵 ───────────────────────────────
    adata_sub = adata[:, common_genes]
    import scipy.sparse as sp
    if sp.issparse(adata_sub.X):
        flex_expr_raw = np.array(adata_sub.X.todense(), dtype=np.float32)
    else:
        flex_expr_raw = np.array(adata_sub.X, dtype=np.float32)

    # ── 8. 细胞类型编码 ───────────────────────────────────────
    cell_types_list = sorted(adata.obs['cell_type'].unique().tolist())
    ct2idx          = {ct: i for i, ct in enumerate(cell_types_list)}
    flex_labels     = np.array([ct2idx[ct] for ct in adata.obs['cell_type']],
                               dtype=np.int64)
    xenium_panel_genes = common_genes

    # ── 9. 保存缓存 ───────────────────────────────────────────
    np.save(EXPORT_FILES['flex_expr'],   flex_expr_raw)
    np.save(EXPORT_FILES['flex_labels'], flex_labels)
    with open(EXPORT_FILES['cell_types'], 'w') as f:
        json.dump(cell_types_list, f)
    with open(EXPORT_FILES['xenium_panel_genes'], 'w') as f:
        json.dump(xenium_panel_genes, f)

    flex_expr = flex_expr_raw
    print('\n  ✅ scRNA 缓存已保存')

# ── 汇总 ───────────────────────────────────────────────────
n_classes    = len(cell_types_list)
gene_list    = xenium_panel_genes
print(f'\nscRNA 矩阵     : {flex_expr.shape[0]:,} × {flex_expr.shape[1]}')
print(f'细胞类型数      : {n_classes}')
print(f'Xenium 基因数   : {len(gene_list)}')
print(f'\n细胞类型列表:')
for i, ct in enumerate(cell_types_list):
    n = (flex_labels == i).sum()
    print(f'  [{i:2d}] {ct:<25} n={n:,}')


In [ ]:
# ============================================================
# Cell 2.5（新增）：肿瘤亚型合并（panel 限制下不可分的类合成一个）
# 在 Cell 2 加载完 flex_labels / cell_types_list 之后插入
# ============================================================
import numpy as np

# 哪些类合并成同一类（基于"Xenium panel 不含 MT/Jun/Fos/VEGFA 等区分基因"判断）
MERGE_GROUPS = {
    "Tumor Cells (Combined)": [
        "Tumor Cells",
        "MT-High, Jun+/Fos+ Tumor Cells",  # MT 基因不在 panel
        "Proliferative Tumor Cells",  # 增殖标志（MKI67 等）通常也不在 panel
        "VEGFA+ Tumor Cells",  # VEGFA 单基因区分不可靠
        "Inflammatory Tumor Cells",
        "Malignant Cells Lining Cyst",
    ],
    # 其余 10 类保持不变
}


def merge_labels(orig_labels, orig_cell_types, merge_groups):
    """
    把 merge_groups 里的类合并到组名下。返回新 labels / 新 cell_types / 旧→新映射表。
    """
    # 旧名 → 新名
    name_map = {}
    for new_name, old_names in merge_groups.items():
        for old in old_names:
            name_map[old] = new_name

    # 构造新类型列表（保留未合并的，加上合并后的新类）
    new_cell_types = []
    seen_new = set()
    for ct in orig_cell_types:
        target = name_map.get(ct, ct)
        if target not in seen_new:
            new_cell_types.append(target)
            seen_new.add(target)

    new_idx_of = {n: i for i, n in enumerate(new_cell_types)}

    # 旧 idx → 新 idx
    idx_map = np.zeros(len(orig_cell_types), dtype=np.int64)
    for old_i, old_name in enumerate(orig_cell_types):
        new_name = name_map.get(old_name, old_name)
        idx_map[old_i] = new_idx_of[new_name]

    new_labels = idx_map[orig_labels]
    return new_labels, new_cell_types, idx_map


flex_labels_orig = flex_labels.copy()
cell_types_orig = list(cell_types_list)

flex_labels, cell_types_list, idx_remap = merge_labels(
    flex_labels_orig,
    cell_types_orig,
    MERGE_GROUPS,
)
n_classes = len(cell_types_list)

print(f"类数: {len(cell_types_orig)} → {n_classes}")
print(f"新类型分布:")
for i, ct in enumerate(cell_types_list):
    n = int((flex_labels == i).sum())
    pct = n / len(flex_labels) * 100
    print(f"  [{i:>2}] {ct:<35} n={n:,}  ({pct:.1f}%)")


## Stage 2 — 转录本 binning + PCA 对齐 + 联合图

加载 Xenium 转录本（~1.5 亿条），按 `bin_size`μm 聚合成 spot；
在 scRNA 上 fit PCA(100)，把 Xenium 投影到同一空间；
最后构建 4 类边的联合图：scRNA 内 kNN + spot 内 kNN + 跨域 MNN + spot 空间 kNN。

**首次运行约 30–60 分钟**；之后从缓存加载约 30 秒。

随后会绘制图 3-1（转录本密度热力图）作为 Stage 2 的可视化产出。


In [ ]:
# ============================================================
# Stage 2.1 — 加载转录本 + PCA 特征对齐 + 构建 spot 图
# ============================================================
from utils_spot import (
    load_xenium_transcripts,
    bin_transcripts_to_spots,
    prepare_features_for_gnn,
    build_spot_graph,
)

GRAPH_OK = (
    all(os.path.exists(p) for p in [GRAPH_FILE, SCALER_FILE, COORDS_FILE, PCA_FILE])
    and not FORCE['graph']
)

if GRAPH_OK:
    print('[Cache HIT] 加载 spot 图...')
    ckpt          = torch.load(GRAPH_FILE, weights_only=False)
    data          = ckpt['data']
    class_weights = ckpt['class_weights']
    split_info    = ckpt['split_info']
    with open(SCALER_FILE, 'rb') as f:
        fitted_scaler = pickle.load(f)
    with open(PCA_FILE, 'rb') as f:
        pca_model = pickle.load(f)
    spot_coords = np.load(COORDS_FILE)

else:
    print('[Cache MISS] 从转录本构建 spot 图...')

    # 1. 加载 Xenium 转录本
    df_tx = load_xenium_transcripts(
        PATHS['raw']['transcripts'],
        gene_list=xenium_panel_genes,
        qv_threshold=PARAMS['qv_threshold'],
    )

    # 1.5. 保存转录本下采样（用于图 3-1，避免重复加载 1.5 亿条）
    n_sample = min(2_000_000, len(df_tx))
    sample_idx = np.random.RandomState(42).choice(
        len(df_tx), n_sample, replace=False
    )
    np.savez_compressed(
        TX_SAMPLE_FILE,
        x=df_tx['x'].values[sample_idx].astype(np.float32),
        y=df_tx['y'].values[sample_idx].astype(np.float32),
        n_total=np.array([len(df_tx)], dtype=np.int64),
    )
    print(f'  转录本下采样保存: {n_sample:,} / {len(df_tx):,} → {TX_SAMPLE_FILE}')

    # 2. 转录本 → spot 矩阵
    spot_expr_raw, spot_coords = bin_transcripts_to_spots(
        df_tx,
        gene_list=xenium_panel_genes,
        bin_size=PARAMS['bin_size'],
        min_transcripts=PARAMS['min_transcripts'],
    )
    print(f'  Spot 节点数: {spot_expr_raw.shape[0]:,}')

    # 3. PCA 跨域对齐
    print('\nPCA 特征对齐...')
    scrna_pca, spot_pca, pca_model = prepare_features_for_gnn(
        scrna_counts    = flex_expr.astype(np.float32),
        spot_counts     = spot_expr_raw,
        n_pca           = PARAMS['n_pca'],
        clip_percentile = PARAMS['pca_clip_pct'],
    )

    # 4. 构图（含 4 类边）
    print('\n构建联合图...')
    data, class_weights, split_info = build_spot_graph(
        scrna_norm           = scrna_pca,
        spot_norm            = spot_pca,
        spot_coords          = spot_coords,
        scrna_labels         = flex_labels.astype(np.int64),
        k_feat               = PARAMS['k_feat'],
        k_spatial            = PARAMS['k_spatial'],
        k_cross              = PARAMS['k_cross'],
        val_ratio            = PARAMS['val_ratio'],
        max_weight_multiplier= PARAMS['max_weight_multiplier'],
    )

    # 5. 保存所有缓存
    torch.save(
        {'data': data, 'class_weights': class_weights, 'split_info': split_info},
        GRAPH_FILE,
    )
    with open(PCA_FILE, 'wb') as f:
        pickle.dump(pca_model, f)
    np.save(COORDS_FILE, spot_coords)

    # fitted_scaler 留给 TopACT 用（如果它选择不直接用 PCA 特征）
    from utils_spot import unified_normalize_spot
    _, _, fitted_scaler = unified_normalize_spot(
        flex_expr.astype(np.float32), spot_expr_raw,
    )
    with open(SCALER_FILE, 'wb') as f:
        pickle.dump(fitted_scaler, f)

    # 释放大对象
    del df_tx, spot_expr_raw, scrna_pca, spot_pca
    import gc; gc.collect()

    print(f'\n✅ 图缓存已保存: {GRAPH_FILE}')

# ── 图状态报告 ────────────────────────────────────────────────
print(
    f"\n节点数      : {data.x.shape[0]:,}  "
    f"(scRNA {split_info['n_scrna']:,} + spot {split_info['n_spots']:,})"
)
print(f"特征维度    : {data.x.shape[1]}  ← PCA {PARAMS['n_pca']}d")
print(f"总边数      : {data.edge_index.shape[1]:,}")
print(f"跨域边占比  : {split_info.get('cross_edge_pct', -1):.1f}%  (健康 5–50%)")
print(f"训练/验证   : {data.train_mask.sum().item():,} / {data.val_mask.sum().item():,}")
print('Stage 2.1 完成 ✓')


In [ ]:
# ============================================================
# Stage 2.2 — 图 3-1 转录本密度热力图
# 数据源：Stage 2.1 保存的下采样转录本（TX_SAMPLE_FILE）
# ============================================================
import matplotlib.pyplot as plt

if not os.path.exists(TX_SAMPLE_FILE):
    print('⚠  转录本下采样文件不存在（图缓存命中时未生成）')
    print(f'   要绘制图 3-1，请设 FORCE["graph"]=True 重跑 Stage 2.1')
else:
    arr = np.load(TX_SAMPLE_FILE)
    x_arr, y_arr = arr['x'], arr['y']
    n_total = int(arr['n_total'][0])
    print(f'  绘制 {len(x_arr):,} 个采样点（总转录本 {n_total:,}）')

    fig, ax = plt.subplots(figsize=(10, 8), dpi=200)
    hb = ax.hexbin(x_arr, y_arr, gridsize=200, cmap='YlOrRd', mincnt=1)
    ax.set_aspect('equal')
    ax.set_xlabel('x (μm)')
    ax.set_ylabel('y (μm)')
    ax.set_title(
        f'Xenium 卵巢癌组织转录本密度  '
        f'(采样 {len(x_arr):,} / 总 {n_total:,}, '
        f'Q≥{PARAMS["qv_threshold"]})'
    )
    plt.colorbar(hb, ax=ax, label='转录本数')
    out = f'{PATHS["figures"]}fig_3_1_transcript_density.png'
    plt.tight_layout(); plt.savefig(out, dpi=200, bbox_inches='tight')
    plt.show()
    print(f'✅ 图 3-1 已保存: {out}')


## Stage 3 — GNN 模型训练

训练 3 个 GNN 架构：GCN / GraphSAGE / GAT。

- **断点续跑**：每个模型权重独立保存，已存在则跳过
- **`FORCE['train']=True`** 强制重训所有模型
- **GAT OOM 不影响其他**：异常被 catch，其他模型继续


In [ ]:
# ============================================================
# Cell 4a: 初始化三种 GNN 模型
# ============================================================
from models_amp import GCN_AMP, GraphSAGE_AMP, GAT_AMP
from utils import run_experiment

in_dim = data.x.shape[1]
device = torch.device(PARAMS['device'])

model_configs = {
    'GCN': GCN_AMP(
        in_channels     = in_dim,
        hidden_channels = PARAMS['hidden_dim'],
        out_channels    = n_classes,
        proj_dim        = PARAMS['proj_dim'],
        dropout         = PARAMS['dropout'],
    ),
    'GraphSAGE': GraphSAGE_AMP(
        in_channels     = in_dim,
        hidden_channels = PARAMS['hidden_dim'],
        out_channels    = n_classes,
        proj_dim        = PARAMS['proj_dim'],
        dropout         = PARAMS['dropout'],
    ),
    'GAT': GAT_AMP(
        in_channels     = in_dim,
        hidden_channels = PARAMS['hidden_dim'],
        out_channels    = n_classes,
        proj_dim        = PARAMS['proj_dim'],
        dropout         = PARAMS['dropout'],
        heads           = PARAMS.get('gat_heads', 2),  # 默认2头，避免OOM
    ),
}
print('模型参数量:')
for name, model in model_configs.items():
    n_params = sum(p.numel() for p in model.parameters())
    print(f'  {name:<12} {n_params:,}')


In [ ]:
# ============================================================
# Cell 4b: 训练 GNN
#
# 逻辑：
#   - 逐模型检查权重文件是否存在
#   - 只训练缺失权重的模型，跳过已有权重的模型
#   - 不同参数配置的权重文件名不同，可并存
#   - GAT 失败不影响 GCN/GraphSAGE 的加载和后续使用
#
# 权重命名格式：{ModelName}_{weight_tag}.pt
#   示例：GCN_h128_kf15_ks10_bin10.pt
#         GAT_h128_kf8_ks5_bin10.pt
# ============================================================

def _weight_file(name):
    """根据当前参数生成权重文件名（包含参数信息，不同配置可共存）"""
    return os.path.join(PATHS['output']['models'], f'{name}_{weight_tag}.pt')

# ── 逐模型检查状态 ────────────────────────────────────────────────────────
print('=== 模型权重状态检查 ===')
model_status = {}
for name in model_configs:
    wf   = _weight_file(name)
    exists = os.path.exists(wf)
    size   = f"{os.path.getsize(wf)/1e6:.1f} MB" if exists else "—"
    model_status[name] = exists
    status_icon = '✅ 已有' if exists else '⬜ 需训练'
    print(f'  {name:<12} {status_icon}  {wf}  {size}')

need_train = [n for n, ok in model_status.items() if not ok]
have_weights = [n for n, ok in model_status.items() if ok]

if FORCE['train']:
    print(f'\n[强制重训] FORCE["train"]=True → 覆盖所有已有权重')
    need_train = list(model_configs.keys())

print(f'\n需要训练: {need_train if need_train else "无（全部已有权重）"}')
print(f'直接加载: {have_weights}')

# ── 加载已有权重 ──────────────────────────────────────────────────────────
gnn_results = {}

# 尝试读取历史 summary（可能包含部分模型的历史指标）
if os.path.exists(RESULTS_SUMMARY_FILE):
    with open(RESULTS_SUMMARY_FILE, 'rb') as f:
        results_summary = pickle.load(f)
    print(f'\n已读取历史 summary: {RESULTS_SUMMARY_FILE}')
else:
    results_summary = {}

for name in have_weights:
    if FORCE['train'] and name in need_train:
        continue   # 强制重训时跳过已有权重的加载
    try:
        model = model_configs[name]
        state = torch.load(_weight_file(name), map_location='cpu', weights_only=True)
        model.load_state_dict(state)
        model.eval()
        hist_summary = results_summary.get(name, {})
        gnn_results[name] = {
            'model'        : model,
            'best_val_f1'  : hist_summary.get('best_val_f1', float('nan')),
            'best_val_acc' : hist_summary.get('best_val_acc', float('nan')),
            'best_epoch'   : hist_summary.get('best_epoch', -1),
            'history'      : hist_summary.get('history', []),
            'pred_indices' : None,   # 推断步骤补充
            'probabilities': None,
            'confidence'   : None,
        }
        print(f'  ✅ {name} 加载完成  Val F1={hist_summary.get("best_val_f1", float("nan")):.4f}')
    except Exception as e:
        print(f'  ❌ {name} 权重加载失败: {e}')
        need_train.append(name)   # 加载失败则重新训练

# ── 训练缺失权重的模型 ────────────────────────────────────────────────────
if need_train:
    print(f'\n开始训练: {need_train}')

    # 训练前先清理显存（避免已加载模型占用导致 OOM）
    for name in have_weights:
        if name in gnn_results and gnn_results[name].get('model') is not None:
            gnn_results[name]['model'] = gnn_results[name]['model'].cpu()
    import gc; gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        print(f'  显存清理完成，当前空闲: '
              f'{torch.cuda.mem_get_info()[0]/1024**3:.1f} GB')

    for model_name in need_train:
        if model_name not in model_configs:
            print(f'  ⚠️  {model_name} 不在 model_configs 中，跳过')
            continue

        print(f'\n{"─"*55}')
        print(f'  训练 {model_name}  (权重将保存为 {_weight_file(model_name)})')
        print(f'{"─"*55}')

        # 重新实例化模型（避免之前训练污染状态）
        from models_amp import GCN_AMP, GraphSAGE_AMP, GAT_AMP
        arch_map = {
            'GCN'       : GCN_AMP,
            'GraphSAGE' : GraphSAGE_AMP,
            'GAT'       : GAT_AMP,
        }
        ModelClass = arch_map[model_name]
        fresh_model = ModelClass(
            in_channels     = in_dim,
            hidden_channels = PARAMS['hidden_dim'],
            out_channels    = n_classes,
            proj_dim        = PARAMS['proj_dim'],
            dropout         = PARAMS['dropout'],
        )

        try:
            result = run_experiment(
                model         = fresh_model,
                data          = data,
                class_weights = class_weights,
                n_classes     = n_classes,
                device        = device,
                params        = PARAMS,
                model_name    = model_name,
                save_dir      = None,   # 不用 models.py 自带的保存，下面手动保存
            )

            # 保存带参数信息的权重文件名
            torch.save(result['model'].state_dict(), _weight_file(model_name))
            print(f'  💾 权重已保存: {_weight_file(model_name)}')

            gnn_results[model_name] = result
            results_summary[model_name] = {
                'best_val_f1'  : result['best_val_f1'],
                'best_val_acc' : result.get('best_val_acc', 0.0),
                'best_epoch'   : result.get('best_epoch', -1),
                'history'      : result.get('history', []),
                'weight_file'  : _weight_file(model_name),
                'weight_tag'   : weight_tag,
            }

            # 训练完立即保存 summary（即使后续模型失败也有记录）
            with open(RESULTS_SUMMARY_FILE, 'wb') as f:
                pickle.dump(results_summary, f)

            print(f'  ✅ {model_name} 训练完成  Val F1={result["best_val_f1"]:.4f}')

        except torch.cuda.OutOfMemoryError as oom:
            print(f'\n  ❌ {model_name} OOM: {oom}')
            print(f'  → 跳过 {model_name}，其他已完成模型不受影响')
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        except Exception as e:
            import traceback
            print(f'\n  ❌ {model_name} 训练异常: {type(e).__name__}: {e}')
            traceback.print_exc()
            print(f'  → 跳过 {model_name}，继续后续模型')
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        # 每个模型训练后将其移回 CPU，释放显存给下一个
        if model_name in gnn_results and gnn_results[model_name].get('model'):
            gnn_results[model_name]['model'] = gnn_results[model_name]['model'].cpu()
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    # 将已有权重模型移回 GPU 状态的模型重新设置（只在需要时）
    for name in have_weights:
        if name in gnn_results and gnn_results[name].get('model') is not None:
            # 保持在 CPU，推断时再按需移动
            pass

# ── 最终汇总 ─────────────────────────────────────────────────────────────
print(f'\n{"="*55}')
print('模型汇总:')
for name in model_configs:
    if name in gnn_results:
        f1 = gnn_results[name].get("best_val_f1", float("nan"))
        wf = os.path.basename(_weight_file(name))
        f1_str = f"{f1:.4f}" if f1 == f1 else "N/A"
        print(f'  ✅ {name:<12} Val F1={f1_str}  ({wf})')
    else:
        print(f'  ❌ {name:<12} 未完成（权重不存在）')
print(f'{"="*55}')
print(f'\n当前 weight_tag: {weight_tag}')
print(f'权重目录: {PATHS["output"]["models"]}')
print('（不同参数配置的权重文件名不同，可安全共存）')


## Stage 4 — 推断 + 基线 + 工具函数

1. GNN 前向推断所有节点（scRNA + spot）的概率，缓存到 `SPOT_PROBAS_FILE`
2. 训练并评估 TopACT (SVM) 基线
3. 定义 helper 函数（`get_val_f1` / `is_gnn`）和全局 `best_name`，供后续所有 cell 复用

执行后，关键变量包括：

- `val_preds[method]`: scRNA 验证集预测（int 数组）
- `spot_probas[method]`: Xenium spot 概率矩阵（含 GNN + TopACT）
- `best_name`: F1 最高的 GNN 模型名（注意：不会选择 TopACT）


In [ ]:
# ============================================================
# Cell 5: 推断所有节点概率 + scRNA 验证集评估
# ============================================================
from eval import compute_metrics

val_idx  = split_info['val_idx']
val_true = flex_labels[val_idx]
n_scrna  = split_info['n_scrna']

EVAL_OK = os.path.exists(SPOT_PROBAS_FILE) and not FORCE['eval']

if EVAL_OK:
    print('[Cache HIT] 加载预测概率缓存...')
    with open(SPOT_PROBAS_FILE, 'rb') as f:
        probas_cache = pickle.load(f)
    val_preds   = probas_cache['val_preds']
    spot_probas = probas_cache['spot_probas']
    # ── 修复：从缓存恢复 spot_coords，确保与 probas 同步 ──────
    if 'spot_coords' in probas_cache:
        spot_coords = probas_cache['spot_coords']
        print(f'  spot_coords 已从缓存恢复: {spot_coords.shape}')
    else:
        # 旧缓存没有 spot_coords，做一次长度校验提示
        _n_proba  = len(next(iter(spot_probas.values())))
        _n_coords = len(spot_coords)
        if _n_proba != _n_coords:
            print(f'⚠️  旧缓存缺少 spot_coords，且长度不匹配 '
                  f'(proba={_n_proba}, coords={_n_coords})。\n'
                  f'   请设置 FORCE["eval"]=True 重新推断以修复。')
        else:
            print(f'  旧缓存兼容，spot_coords 长度一致: {_n_coords}')

else:
    print('[Cache MISS] 前向推断所有节点...')
    val_preds   = {}
    spot_probas = {}
    data_cpu    = data.cpu()

    for name, res in gnn_results.items():
        model_eval = res['model'].to('cpu').eval()
        with torch.no_grad():
            log_probs = model_eval(data_cpu)
            proba_all = torch.softmax(log_probs, dim=1).numpy()
        val_preds[name]   = proba_all[val_idx].argmax(axis=1)
        spot_probas[name] = proba_all[n_scrna:]
        print(f'  {name} 推断完成  spot_proba shape: {spot_probas[name].shape}')

    # ── 修复：同时保存 spot_coords，防止缓存不同步 ──────────────
    with open(SPOT_PROBAS_FILE, 'wb') as f:
        pickle.dump({
            'val_preds'  : val_preds,
            'spot_probas': spot_probas,
            'spot_coords': spot_coords,   # ← 新增：绑定坐标与预测
        }, f)
    print(f'✅ 预测概率 + spot_coords 已保存 → {SPOT_PROBAS_FILE}')
    print(f'   spot_coords shape: {spot_coords.shape}')

# ── 长度最终校验（无论 HIT 还是 MISS 都检查）──────────────────
_first_proba = next(iter(spot_probas.values()))
assert len(_first_proba) == len(spot_coords), (
    f"spot_probas ({len(_first_proba)}) 与 spot_coords ({len(spot_coords)}) "
    f"长度仍不一致，请删除缓存文件后重新运行:\n  {SPOT_PROBAS_FILE}"
)
print(f'\n✅ 长度校验通过: spot_probas = spot_coords = {len(spot_coords):,} spots')

print('\n' + '=' * 65)
print('  scRNA 验证集评估（Ground Truth = Flex 预标注细胞类型）')
print('=' * 65)
print(f'{"Method":<14} {"Acc":>7} {"F1-Mac":>8} {"F1-Wei":>8} {"Kappa":>8}')
print('-' * 65)
for method, y_pred in val_preds.items():
    m = compute_metrics(val_true, y_pred, n_classes=n_classes)
    print(f'{method:<14} {m["accuracy"]:>7.4f} {m["f1_macro"]:>8.4f} '
          f'{m["f1_weighted"]:>8.4f} {m["kappa"]:>8.4f}')
print('=' * 65)


In [ ]:
# ============================================================
# Cell 5b（新增）: TopACT 基线（SVM）
#   - 训练：scRNA 训练集（与 GNN 完全相同的划分 train_idx）
#   - 评估：scRNA 验证集（与 GNN 完全相同的 val_idx，公平对比）
#   - 推断：Xenium spot（带空间平滑），用于后续 Moran's I 对比
# ============================================================
import time, pickle, os
import numpy as np
from sklearn.metrics import (accuracy_score, f1_score, cohen_kappa_score,
                             confusion_matrix)
from topact import TopACT

TOPACT_PROBA_FILE  = 'results/predictions/topact_spot_proba.pkl'
TOPACT_METRIC_FILE = 'results/evaluation/topact_metrics.pkl'

# ── 1. 取出训练用数据（与 GNN 完全相同的划分）─────────────
# data.x[:n_scrna] 是 scRNA 节点的（PCA 后的）特征，flex_labels 是整型标签
n_scrna   = split_info['n_scrna']
train_idx = split_info['train_idx']
val_idx   = split_info['val_idx']

X_scrna   = data.x[:n_scrna].cpu().numpy()       # ← 与 GNN 同源特征
X_train   = X_scrna[train_idx]
y_train   = flex_labels[train_idx]
X_val     = X_scrna[val_idx]
y_val     = flex_labels[val_idx]
print(f'TopACT 训练集 {X_train.shape}, 验证集 {X_val.shape}')

# ── 2. 训练 SVM（不再 fit scaler，因为 PCA 已对齐过分布）────
# 注意：这里 fitted_scaler=None 让 TopACT 内部 fit 一个新的 scaler
#       因为 data.x 已经是 PCA 特征，再做 StandardScaler 是无害的
t0 = time.time()
topact = TopACT(C=1.0, gamma="scale", kernel="rbf",
                spatial_weight=0.3, n_neighbors=10, seed=42)
topact.fit(X_train, y_train, fitted_scaler=None)
print(f'  SVM fit 完成 ({time.time()-t0:.1f}s)')

# ── 3. scRNA 验证集评估 ───────────────────────────────────
t0 = time.time()
val_pred = topact.predict(X_val, spatial_coords=None)   # 验证集无空间坐标
print(f'  验证集推断完成 ({time.time()-t0:.1f}s)')

acc   = accuracy_score(y_val, val_pred)
f1m   = f1_score(y_val, val_pred, average='macro')
f1w   = f1_score(y_val, val_pred, average='weighted')
kappa = cohen_kappa_score(y_val, val_pred)
print(f'  TopACT  Acc={acc:.4f}  F1-Mac={f1m:.4f}  '
      f'F1-Wei={f1w:.4f}  Kappa={kappa:.4f}')

# ── 4. Xenium spot 推断（带空间平滑）──────────────────────
X_spot = data.x[n_scrna:].cpu().numpy()
t0 = time.time()
spot_pred, spot_proba_topact = topact.predict(
    X_spot, spatial_coords=spot_coords, return_proba=True,
)
print(f'  Spot 推断完成 ({time.time()-t0:.1f}s)  shape={spot_proba_topact.shape}')

# ── 5. 缓存（避免重跑）─────────────────────────────────────
os.makedirs(os.path.dirname(TOPACT_METRIC_FILE), exist_ok=True)
with open(TOPACT_PROBA_FILE, 'wb') as f:
    pickle.dump({'spot_proba': spot_proba_topact,
                 'spot_pred': spot_pred,
                 'spot_coords': spot_coords}, f)
with open(TOPACT_METRIC_FILE, 'wb') as f:
    pickle.dump({'acc': acc, 'f1_macro': f1m,
                 'f1_weighted': f1w, 'kappa': kappa,
                 'val_pred': val_pred, 'y_val': y_val}, f)
print(f'  缓存完成 → {TOPACT_PROBA_FILE}, {TOPACT_METRIC_FILE}')

# ── 6. 把 TopACT 加入到主对比表 ────────────────────────────
spot_probas['TopACT'] = spot_proba_topact   # ← 让后续 Cell 6/7/8/9 自动包含 TopACT
val_preds  ['TopACT'] = val_pred
print('TopACT 已加入到对比表')


In [ ]:
# ============================================================
# Stage 4.3 — 全局 helper 函数 + best_name 统一定义
# 后续所有 cell 都用这里定义的：
#   - get_val_f1(name)  : 任何方法（GNN/TopACT/...）的 val F1，永远不会 KeyError
#   - is_gnn(name)      : 区分 GNN 与基线
#   - best_name         : 最优 GNN 模型名（用于挑选混淆矩阵、Moran's I 等的代表模型）
# ============================================================
import os, pickle
from sklearn.metrics import f1_score


def get_val_f1(method_name: str) -> float:
    """统一查 val F1。优先级：gnn_results → TopACT 缓存 → 实时算。"""
    if method_name in gnn_results:
        v = gnn_results[method_name].get('best_val_f1')
        if v is not None and v == v:        # NaN safe
            return float(v)
    if method_name == 'TopACT' and os.path.exists(TOPACT_METRIC_FILE):
        with open(TOPACT_METRIC_FILE, 'rb') as fp:
            return float(pickle.load(fp)['f1_macro'])
    if method_name in val_preds:
        y_val = flex_labels[split_info['val_idx']]
        return float(f1_score(y_val, val_preds[method_name], average='macro'))
    return float('nan')


def is_gnn(method_name: str) -> bool:
    """是否为 GNN 方法（用于显示和最优选择）。"""
    return method_name in gnn_results


# ── 选 best_name：只在 GNN 中选，不能是 TopACT ─────────────────
if not gnn_results:
    raise RuntimeError('gnn_results 为空，无法选择 best_name。请先跑 Stage 3 训练。')

best_name = max(
    gnn_results.keys(),
    key=lambda n: gnn_results[n].get('best_val_f1', 0) or 0,
)
best_proba = spot_probas[best_name]

print(f'✅ best_name = {best_name}  (val F1 = {get_val_f1(best_name):.4f})')

# ── 自检：所有方法清单 ────────────────────────────────────────
print('\n方法清单与 val F1:')
for n in spot_probas.keys():
    tag = 'GNN' if is_gnn(n) else 'baseline'
    star = '  ★ best' if n == best_name else ''
    print(f'  [{tag:<8}] {n:<12} val_f1 = {get_val_f1(n):.4f}{star}')


## Stage 5 — 诊断

两组诊断：

1. **塌缩诊断**：scRNA vs spot 端类分布对比，标记被压成 ~0% 的大类
2. **P0 修复验证**：跨域边占比 + Tumor / MT-High 类分布检查

注意：合并肿瘤亚型后 P0 验证会自动用合并后的类（"Tumor Cells (Combined)"）做检查。


In [ ]:
# ============================================================
# Cell DIAG: 塌缩诊断（在 Cell 5 之后运行）
# ============================================================
import numpy as np

print("=" * 70)
print("  GNN 输出塌缩诊断")
print("=" * 70)

# 1. 类分布对比
ref_dist = np.bincount(flex_labels, minlength=n_classes) / len(flex_labels)
pred_dists = {
    name: np.bincount(p.argmax(1), minlength=n_classes) / len(p)
    for name, p in spot_probas.items()
}

header = f"\n{'Cell type':<35}{'scRNA':>9}"
for name in spot_probas:
    header += f"{name:>11}"
print(header)
print("-" * len(header))

n_collapsed = {name: 0 for name in spot_probas}
for i, ct in enumerate(cell_types_list):
    line = f"{ct[:34]:<35}{ref_dist[i]:>8.1%} "
    for name in spot_probas:
        d = pred_dists[name][i]
        is_col = ref_dist[i] > 0.05 and d < 0.005
        if is_col:
            n_collapsed[name] += 1
        line += f"{d:>9.1%}{'⚠' if is_col else ' '} "
    print(line)
print(f"\n  塌缩类计数（大类被压成 ~0%）：{n_collapsed}  健康 0；>2 即塌缩")

# 2. 跨域边占比（应与 Cell 3 输出一致，<5% 警告）
ei = data.edge_index.cpu().numpy()
n_sc = split_info["n_scrna"]
cross = int(((ei[0] < n_sc) ^ (ei[1] < n_sc)).sum())
total = ei.shape[1]
print(f"\n跨域边占比: {cross:,} / {total:,} = {100*cross/total:.2f}%  " f"(健康 5–50%)")

# 3. spot 端 softmax 熵
print(f"\n{'模型':<14}{'mean_entropy':>15}{'top1.mean':>11}{'top1.std':>11}")
print("-" * 51)
H_h = np.log(n_classes) / 2
for name, p in spot_probas.items():
    eps = 1e-12
    H = float(-(p * np.log(p + eps)).sum(axis=1).mean())
    top1 = p.max(axis=1)
    health = "✅" if H > H_h * 0.5 else ("⚠️ 塌缩" if H < 0.5 else "❓")
    print(f"{name:<14}{H:>15.4f}{top1.mean():>11.4f}{top1.std():>11.4f}  {health}")
print(f"\n  健康 H ≈ {H_h:.2f}（log({n_classes})/2）；H<0.5 + top1>0.95 = 塌缩")


In [ ]:
# ============================================================
# Stage 5.2 — P0 修复验证（防御式查找；兼容合并/未合并两种情况）
# 通过/失败标准：
#   ✅ 跨域边占比 ∈ [5%, 50%]   （目标 ~15–25%）
#   ✅ 主要肿瘤类预测占比 与真实占比相差 ≤ 5pp
# ============================================================
import numpy as np
from collections import Counter

# 1. 跨域边
cross_pct = split_info.get('cross_edge_pct', -1)
status_a = "✅" if 5 <= cross_pct <= 50 else "❌"
print(f"[1/2] 跨域边占比: {cross_pct:.1f}%  {status_a}")

# 2. 关键塌缩类的预测占比 vs 真实占比
#    防御式查找：合并后只剩 'Tumor Cells (Combined)'；未合并则有 'Tumor Cells' / 'MT-High...'
target_classes = []
for keyword in ['Tumor Cells (Combined)', 'Tumor Cells', 'MT-High',
                'Proliferative Tumor']:
    for i, ct in enumerate(cell_types_list):
        if keyword in ct and i not in [t[0] for t in target_classes]:
            target_classes.append((i, ct))
            break
    if len(target_classes) >= 3:
        break

if not target_classes:
    print('[2/2] 未找到肿瘤类，跳过塌缩验证')
else:
    preds   = best_proba.argmax(axis=1)
    counts  = Counter(preds.tolist())
    n_total = len(preds)
    ref_dist = np.bincount(flex_labels, minlength=n_classes) / len(flex_labels)

    all_pass = True
    print(f"[2/2] 主要肿瘤类预测占比 vs 真实占比 (使用 {best_name})：")
    for cls_idx, cls_name in target_classes:
        true_pct = ref_dist[cls_idx] * 100
        pred_pct = counts.get(cls_idx, 0) / n_total * 100
        diff = abs(pred_pct - true_pct)
        ok = diff <= 5.0           # 允许 5pp 偏差
        all_pass = all_pass and ok
        status = "✅" if ok else "❌"
        print(f'   {cls_name[:35]:<35}  '
              f'真实 {true_pct:>5.1f}%  预测 {pred_pct:>5.1f}%  '
              f'(Δ={pred_pct-true_pct:+.1f}pp)  {status}')

    if status_a == "✅" and all_pass:
        print("\n🎉 P0 全部通过：跨域边健康 + 类分布合理")
    elif not all_pass:
        print("\n⚠  仍有类分布偏差，但模型可能仍可用于论文：")
        print("   → 如偏差在小类上，问题可能是 panel 不可分（合并后已部分缓解）")
        print("   → 如偏差在大类上，考虑提升 lambda_mmd 或重新检查 PCA 对齐")


## Stage 6 — 保存预测 + 空间可视化

1. 保存最优 GNN 模型的 spot 级预测为 CSV，所有方法的完整概率矩阵也分别导出
2. 绘制所有方法的空间预测对比图（GCN / GraphSAGE / GAT / TopACT 并排）


In [ ]:
# ============================================================
# Stage 6.1 — 保存 spot 级别预测结果（CSV）
# best_name 来自 Stage 4.3 的 helper cell
# ============================================================
print(f'最优 GNN 模型: {best_name}  Val F1={get_val_f1(best_name):.4f}')

# 长度校验
n_proba  = len(best_proba)
n_coords = len(spot_coords)
if n_proba != n_coords:
    raise ValueError(
        f"spot_probas ({n_proba}) 与 spot_coords ({n_coords}) 长度不一致。\n"
        f"请设置 FORCE['eval']=True 并重跑 Stage 4."
    )
print(f'spot 数量校验通过: {n_coords:,}')

# ── 主输出（最优模型）─────────────────────────────────────────
best_pred_idx   = best_proba.argmax(axis=1)
best_pred_label = [cell_types_list[i] for i in best_pred_idx]
best_confidence = best_proba.max(axis=1)

spot_df = pd.DataFrame({
    'x'         : spot_coords[:, 0],
    'y'         : spot_coords[:, 1],
    'pred_idx'  : best_pred_idx,
    'pred_label': best_pred_label,
    'confidence': best_confidence,
})
out_csv = os.path.join(PATHS['output']['predictions'],
                       f'spot_predictions_{best_name}.csv')
spot_df.to_csv(out_csv, index=False)
print(f'最优模型 spot 预测已保存: {out_csv}  ({len(spot_df):,} spots)')

# ── 全部方法（含 TopACT）──────────────────────────────────────
for name, proba in spot_probas.items():
    df_all = pd.DataFrame(
        proba, columns=[f'prob_{c}' for c in cell_types_list]
    )
    df_all['x']          = spot_coords[:, 0]
    df_all['y']          = spot_coords[:, 1]
    df_all['pred_label'] = [cell_types_list[i] for i in proba.argmax(axis=1)]
    out_full = os.path.join(PATHS['output']['predictions'],
                            f'spot_predictions_{name}_full.csv')
    df_all.to_csv(out_full, index=False)
print('所有方法 spot 预测已保存 ✓')


In [ ]:
# ============================================================
# Stage 6.2 — 空间可视化（所有方法并排）
# ============================================================
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# 显示顺序：GNN 在前，基线在后
ordered = [n for n in spot_probas if is_gnn(n)] + \
          [n for n in spot_probas if not is_gnn(n)]
n_panels = len(ordered)

fig, axes = plt.subplots(1, n_panels, figsize=(7*n_panels, 7))
if n_panels == 1:
    axes = [axes]

cmap   = plt.cm.get_cmap('tab20', n_classes)
colors = {i: cmap(i) for i in range(n_classes)}

for ax, model_name in zip(axes, ordered):
    proba    = spot_probas[model_name]
    pred_idx = proba.argmax(axis=1)
    c_vals   = [colors[i] for i in pred_idx]
    ax.scatter(spot_coords[:, 0], spot_coords[:, 1],
               c=c_vals, s=0.3, alpha=0.5, rasterized=True)
    ax.set_aspect('equal')
    method_tag = 'GNN' if is_gnn(model_name) else 'Baseline'
    ax.set_title(
        f'{model_name} ({method_tag} Spot-level)\n'
        f'Val F1={get_val_f1(model_name):.3f}',
        fontsize=11,
    )
    ax.set_xlabel('x (μm)')
    ax.set_ylabel('y (μm)')

patches = [mpatches.Patch(color=colors[i], label=cell_types_list[i])
           for i in range(n_classes)]
fig.legend(handles=patches, bbox_to_anchor=(1.01, 0.9),
           loc='upper left', fontsize=8, framealpha=0.8)
plt.suptitle(
    f'Spot-level Cell Type Predictions  '
    f'(Ovarian Cancer Xenium, bin={PARAMS["bin_size"]}μm, no segmentation)',
    fontsize=13, y=1.02,
)
plt.tight_layout()
out_fig = os.path.join(PATHS['figures'], 'spot_spatial_all_models.png')
plt.savefig(out_fig, dpi=150, bbox_inches='tight')
plt.show()
print(f'✅ 空间预测图已保存: {out_fig}')


## Stage 7 — Moran's I 空间自相关

对所有方法（GNN + TopACT + Random baseline）计算全局 Moran's I，
作为空间一致性的定量指标。结果保存到 `MORAN_FILE`，供后续表 5-5 和图 5-5 使用。


In [ ]:
# ============================================================
# Stage 7 — Moran's I 空间自相关（所有方法 + Random 基线）
# ============================================================
from topact import TopACT

moran_results = {}

# 1. 所有方法（GNN + TopACT）
print("计算各方法的全局 Moran's I ...")
for method_name, proba in spot_probas.items():
    pred_int = proba.argmax(axis=1)
    moran_results[method_name] = TopACT.morans_i(
        labels=pred_int, coords=spot_coords, n_neighbors=10,
    )
    m = moran_results[method_name]
    print(f"  {method_name:<12} I={m['I']:.4f}  z={m['z_score']:.2f}  "
          f"p={m['p_value']:.2e}")

# 2. Random baseline 用于对比
n_spots = len(spot_coords)
np.random.seed(42)
random_pred = np.random.randint(0, n_classes, size=n_spots)
moran_results['Random'] = TopACT.morans_i(
    labels=random_pred, coords=spot_coords, n_neighbors=10,
)
m = moran_results['Random']
print(f"  {'Random':<12} I={m['I']:.4f}  z={m['z_score']:.2f}  "
      f"p={m['p_value']:.2e}  (基线)")

# 3. 保存
os.makedirs(os.path.dirname(MORAN_FILE), exist_ok=True)
with open(MORAN_FILE, 'wb') as f:
    pickle.dump(moran_results, f)
print(f"\n✅ Moran's I 全方法结果已保存: {MORAN_FILE}")


## Stage 8 — Seurat 标签转移对比

对比 GNN spot 级别预测 vs Seurat TransferData 细胞级别预测。

**前置依赖**：先在 `labelTransfer_ovarian.ipynb` 跑出 Seurat 预测 CSV。
若文件不存在，本 stage 会跳过 Seurat 部分但仍展示 GNN 结果。

聚合方法：用 `aggregate_spot_to_cell` 把 spot 概率按半径聚合到 Seurat 细胞质心，
然后做细胞级一致性比较。


In [ ]:
# ============================================================
# Stage 8 — Seurat 标签转移 vs GNN 对比分析
#   - GNN: spot 级别（不依赖细胞分割）
#   - Seurat TransferData: 细胞级别（依赖 Cellpose 分割）
#
# 三种对比：
#   1. 空间分布并排图（4-panel：GNN 方法 + Seurat）
#   2. 类型比例堆叠条形图（含 scRNA 参考分布）
#   3. GNN spot → 细胞质心聚合，与 Seurat 细胞级一致性
# ============================================================
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from utils_spot import aggregate_spot_to_cell

SEURAT_LT_FILE = os.path.join(
    PATHS['output']['predictions'],
    'seurat_label_transfer_ovarian_enhanced.csv',
)

if not os.path.exists(SEURAT_LT_FILE):
    print(f'⚠  Seurat 结果不存在: {SEURAT_LT_FILE}')
    print('   请先运行 labelTransfer_ovarian.ipynb，再回来跑此 cell')
    print('   本次跳过 Seurat 部分。')
    has_seurat = False
    seurat_df  = None
else:
    seurat_df  = pd.read_csv(SEURAT_LT_FILE)
    has_seurat = True
    print(f'Seurat 加载完成: {len(seurat_df):,} 细胞')
    print(f'  预测类型数: {seurat_df["predicted_id"].nunique()}')
    print(f'  平均置信度: {seurat_df["prediction_score"].mean():.3f}')

# ── 1. 空间分布并排图 ─────────────────────────────────────────
print('\n绘制空间对比图...')

# 统一颜色映射
all_types = sorted(set(cell_types_list))
if has_seurat:
    seurat_types = seurat_df['predicted_id'].dropna().unique().tolist()
    all_types    = sorted(set(all_types) | set(seurat_types))
cmap       = plt.cm.get_cmap('tab20', len(all_types))
type2color = {t: cmap(i) for i, t in enumerate(all_types)}

ordered_methods = [n for n in spot_probas if is_gnn(n)] + \
                  [n for n in spot_probas if not is_gnn(n)]
n_panels = len(ordered_methods) + (1 if has_seurat else 0)

fig, axes = plt.subplots(1, n_panels, figsize=(7 * n_panels, 7))
if n_panels == 1:
    axes = [axes]

# 各方法（spot 级别）
for ax, model_name in zip(axes, ordered_methods):
    proba       = spot_probas[model_name]
    pred_labels = [cell_types_list[i] for i in proba.argmax(axis=1)]
    c_vals      = [type2color[l] for l in pred_labels]
    ax.scatter(spot_coords[:, 0], spot_coords[:, 1],
               c=c_vals, s=0.3, alpha=0.6, rasterized=True)
    ax.set_aspect('equal')
    tag = 'GNN' if is_gnn(model_name) else 'Baseline'
    ax.set_title(
        f'{model_name} ({tag} Spot-level)\n'
        f'Val F1={get_val_f1(model_name):.3f}',
        fontsize=11,
    )
    ax.set_xlabel('x (μm)'); ax.set_ylabel('y (μm)')

# Seurat（细胞级别）
if has_seurat:
    ax = axes[-1]
    valid  = seurat_df['predicted_id'].notna()
    c_vals = [type2color.get(t, (0.5, 0.5, 0.5, 1))
              for t in seurat_df.loc[valid, 'predicted_id']]
    ax.scatter(
        seurat_df.loc[valid, 'x'], seurat_df.loc[valid, 'y'],
        c=c_vals, s=1.0, alpha=0.6, rasterized=True,
    )
    ax.set_aspect('equal')
    ax.set_title(
        f'Seurat TransferData (Cell-level)\n'
        f'Mean Score={seurat_df["prediction_score"].mean():.3f}  '
        f'★ Requires Cell Segmentation',
        fontsize=11,
    )
    ax.set_xlabel('x (μm)'); ax.set_ylabel('y (μm)')

patches = [mpatches.Patch(color=type2color[t], label=t) for t in all_types]
fig.legend(handles=patches, bbox_to_anchor=(1.01, 0.95),
           loc='upper left', fontsize=8, framealpha=0.8, title='Cell Type')
fig.suptitle(
    'Spatial Cell Type Prediction: GNN (no segmentation) vs Seurat (segmentation)',
    fontsize=12, y=1.02,
)
plt.tight_layout()
out_compare = os.path.join(PATHS['figures'], 'spatial_comparison_gnn_vs_seurat.png')
plt.savefig(out_compare, dpi=150, bbox_inches='tight')
plt.show()
print(f'✅ 空间对比图已保存: {out_compare}')

# ── 2. 类型比例堆叠条形图 ─────────────────────────────────────
print('\n绘制类型比例对比...')
sources = {}

# scRNA 参考
n_per_type = np.bincount(flex_labels, minlength=n_classes)
total_sc   = n_per_type.sum()
sources['scRNA (ref)'] = {
    t: n_per_type[i] / total_sc for i, t in enumerate(cell_types_list)
}
# 各方法
for method_name, proba in spot_probas.items():
    pred_idx = proba.argmax(axis=1)
    n_per    = np.bincount(pred_idx, minlength=n_classes)
    sources[method_name] = {
        t: n_per[i] / n_per.sum() for i, t in enumerate(cell_types_list)
    }
# Seurat
if has_seurat:
    vc    = seurat_df['predicted_id'].value_counts()
    total = vc.sum()
    sources['Seurat'] = {t: vc.get(t, 0) / total for t in all_types}

prop_df    = pd.DataFrame(sources, index=all_types).fillna(0).T
bar_colors = [type2color[t] for t in all_types]

fig, ax = plt.subplots(figsize=(max(10, len(sources) * 2), 5))
prop_df.plot(kind='bar', stacked=True, color=bar_colors,
             ax=ax, width=0.7, legend=False)
ax.set_ylabel('Proportion'); ax.set_ylim(0, 1)
ax.set_title('Cell Type Proportion: scRNA ref vs GNN vs Seurat')
plt.setp(ax.get_xticklabels(), rotation=30, ha='right')
handles = [mpatches.Patch(color=bar_colors[i], label=t)
           for i, t in enumerate(all_types)]
ax.legend(handles=handles, bbox_to_anchor=(1.02, 1),
          loc='upper left', fontsize=8, frameon=False)
fig.tight_layout()
out_prop = os.path.join(PATHS['figures'], 'proportion_comparison.png')
fig.savefig(out_prop, dpi=150, bbox_inches='tight')
plt.show()
print(f'✅ 比例对比图已保存: {out_prop}')

# ── 3. GNN spot → 细胞级一致性 ────────────────────────────────
if has_seurat:
    print('\n聚合 GNN spot 预测到细胞质心...')
    cell_pred_idx, cell_pred_conf = aggregate_spot_to_cell(
        spot_proba  = best_proba,
        spot_coords = spot_coords,
        cell_coords = seurat_df[['x', 'y']].values,
        n_classes   = n_classes,
        bin_size    = PARAMS['bin_size'],
        radius_um   = None,
    )
    gnn_cell_labels       = [cell_types_list[i] for i in cell_pred_idx]
    seurat_labels_aligned = seurat_df['predicted_id'].fillna('Unknown').values

    agree = np.array(gnn_cell_labels) == seurat_labels_aligned
    agreement_rate = agree.mean()
    print(f'\nGNN ({best_name}) vs Seurat 细胞级一致性: {agreement_rate:.1%}')
    print('（两者均无 ground truth，一致率反映两方法的相对一致程度）')

    print('\n各细胞类型一致性：')
    agree_df = pd.DataFrame({
        'seurat'    : seurat_labels_aligned,
        'gnn'       : gnn_cell_labels,
        'agree'     : agree,
        'confidence': cell_pred_conf,
    })
    per_type = agree_df.groupby('seurat')['agree'].agg(['mean', 'count'])
    per_type.columns = ['agreement', 'n_cells']
    per_type = per_type.sort_values('agreement', ascending=False)
    print(per_type.to_string())

    compare_csv = os.path.join(PATHS['output']['evaluation'],
                               'gnn_vs_seurat_cell_comparison.csv')
    agree_df.to_csv(compare_csv, index=False)
    print(f'\n✅ 细胞级对比 CSV 已保存: {compare_csv}')

print('\nStage 8 完成 ✓')


## Stage 9 — 消融实验（论文表 5-2）

5 个配置的消融对比，基于 GraphSAGE（最稳定可训练的架构）：

| 配置 | λ_MMD | λ_Ent | λ_PL | 含义 |
|---|---|---|---|---|
| full   | 0.10 | 0.01 | 0.30 | 完整模型 |
| no_pl  | 0.10 | 0.01 | 0.00 | 去掉伪标签 |
| no_ent | 0.10 | 0.00 | 0.30 | 去掉熵正则 |
| no_mmd | 0.00 | 0.01 | 0.30 | 去掉 MMD |
| no_da  | 0.00 | 0.00 | 0.00 | 去掉所有 DA |

**典型耗时**：5 × 15 min ≈ 75 min（A800 80GB）。

**两阶段评估**：
1. scRNA 验证集（标准 F1/Acc/Kappa） — 直接从消融训练拿到
2. **Xenium 端无监督指标**（KL 散度 / Moran's I / 平均置信度） — 因为消融对 spot 端的影响在 scRNA 验证集上看不出来

**断点续跑**：每个配置训完立即写盘，中断后重跑会自动跳过已完成项。
设 `FORCE['ablation']=True` 强制重跑。

**完整跳过**：在 Stage 0 设 `RUN_ABLATION=False` 可整个 stage 跳过。


In [ ]:
# ============================================================
# Stage 9.1 — 消融实验调度
# 受 Stage 0 的 RUN_ABLATION 开关控制；False 时整段跳过
# ============================================================
import copy, gc, time
from sklearn.metrics import accuracy_score, f1_score, cohen_kappa_score
from utils import run_experiment
from models_amp import GraphSAGE_AMP

# 锁定基础架构（消融只改 lambda_*）
ABL_MODEL_NAME  = 'GraphSAGE'
ABL_MODEL_CLASS = GraphSAGE_AMP

ABLATION_CONFIGS = [
    {'name': 'full',   'lambda_mmd': 0.1, 'lambda_ent': 0.01, 'lambda_pl': 0.3},
    {'name': 'no_pl',  'lambda_mmd': 0.1, 'lambda_ent': 0.01, 'lambda_pl': 0.0},
    {'name': 'no_ent', 'lambda_mmd': 0.1, 'lambda_ent': 0.0,  'lambda_pl': 0.3},
    {'name': 'no_mmd', 'lambda_mmd': 0.0, 'lambda_ent': 0.01, 'lambda_pl': 0.3},
    {'name': 'no_da',  'lambda_mmd': 0.0, 'lambda_ent': 0.0,  'lambda_pl': 0.0},
]

if not RUN_ABLATION:
    print('⏭  RUN_ABLATION=False，跳过 Stage 9.1（消融训练）')
    print('   要跑消融，把 Stage 0 的 RUN_ABLATION 改为 True 后重跑此 cell')
    # 创建占位字典，Stage 9.2 / 10 不会崩
    ablation_results = {}
    if os.path.exists(ABL_RESULTS_FILE):
        with open(ABL_RESULTS_FILE, 'rb') as f:
            ablation_results = pickle.load(f)
        print(f'   但读取了历史结果: {list(ablation_results.keys())}')
else:
    # 加载已有结果（断点续跑）
    if os.path.exists(ABL_RESULTS_FILE):
        with open(ABL_RESULTS_FILE, 'rb') as f:
            ablation_results = pickle.load(f)
        print(f'[续跑] 已加载历史: {list(ablation_results.keys())}')
    else:
        ablation_results = {}

    val_idx = split_info['val_idx']
    y_val   = flex_labels[val_idx]
    n_scrna = split_info['n_scrna']
    in_dim  = data.x.shape[1]
    device  = torch.device(PARAMS['device'])

    for cfg in ABLATION_CONFIGS:
        name = cfg['name']
        if name in ablation_results and not FORCE.get('ablation', False):
            print(f"\n⏭  [{name}] 已存在，跳过 (FORCE['ablation']=True 强制重跑)")
            continue

        # 当前配置 PARAMS（深拷贝防污染）
        p = copy.deepcopy(PARAMS)
        p['lambda_mmd'] = cfg['lambda_mmd']
        p['lambda_ent'] = cfg['lambda_ent']
        p['lambda_pl']  = cfg['lambda_pl']

        print(f"\n{'='*60}")
        print(f"  [{name}]  λ_mmd={cfg['lambda_mmd']}  "
              f"λ_ent={cfg['lambda_ent']}  λ_pl={cfg['lambda_pl']}")
        print(f"{'='*60}")

        fresh_model = ABL_MODEL_CLASS(
            in_channels     = in_dim,
            hidden_channels = PARAMS['hidden_dim'],
            out_channels    = n_classes,
            proj_dim        = PARAMS['proj_dim'],
            dropout         = PARAMS['dropout'],
        )

        t0 = time.time()
        try:
            result = run_experiment(
                model         = fresh_model,
                data          = data,
                class_weights = class_weights,
                n_classes     = n_classes,
                device        = device,
                params        = p,
                model_name    = f'{ABL_MODEL_NAME}_{name}',
                save_dir      = None,
            )
        except torch.cuda.OutOfMemoryError as e:
            print(f"  ❌ OOM: {e}")
            torch.cuda.empty_cache(); gc.collect()
            continue

        # 验证集 + spot 端推断（spot_proba 留给 Stage 9.2 用）
        model_eval = result['model'].to('cpu').eval()
        with torch.no_grad():
            log_probs = model_eval(data.cpu())
            proba_full = torch.softmax(log_probs, dim=1).numpy()
        val_pred   = proba_full[val_idx].argmax(axis=1)
        spot_proba = proba_full[n_scrna:]

        metrics = {
            'name'       : name,
            'lambda_mmd' : cfg['lambda_mmd'],
            'lambda_ent' : cfg['lambda_ent'],
            'lambda_pl'  : cfg['lambda_pl'],
            'accuracy'   : float(accuracy_score(y_val, val_pred)),
            'f1_macro'   : float(f1_score(y_val, val_pred, average='macro')),
            'f1_weighted': float(f1_score(y_val, val_pred, average='weighted')),
            'kappa'      : float(cohen_kappa_score(y_val, val_pred)),
            'best_val_f1': result['best_val_f1'],
            'best_epoch' : result.get('best_epoch', -1),
            'history'    : result.get('history', []),
            'train_time' : time.time() - t0,
            'spot_proba' : spot_proba,    # ← 必存，供 Stage 9.2 算 KL/Moran/Conf
        }
        ablation_results[name] = metrics

        # 立即持久化（防中断丢结果）
        with open(ABL_RESULTS_FILE, 'wb') as f:
            pickle.dump(ablation_results, f)

        print(f"  ✅ [{name}]  Acc={metrics['accuracy']:.4f}  "
              f"F1-Mac={metrics['f1_macro']:.4f}  "
              f"Kappa={metrics['kappa']:.4f}  "
              f"({metrics['train_time']/60:.1f} min)")

        # 释放显存
        del fresh_model, result, model_eval, log_probs, proba_full
        torch.cuda.empty_cache(); gc.collect()

    print(f"\n{'='*60}")
    print(f"消融训练完成。配置: {list(ablation_results.keys())}")
    print(f"结果文件: {ABL_RESULTS_FILE}")
    print(f"{'='*60}")


In [ ]:
# ============================================================
# Stage 9.2 — 消融实验的 Xenium 端无监督评估
# scRNA val F1 看不出 DA 效果（DA 只作用于 spot 端），
# 所以这里加 3 个 Xenium 端代理指标：
#   1. KL(predict_dist || scrna_dist)  ← DA 应让两者更接近
#   2. 全局 Moran's I                  ← DA 应提升空间一致性
#   3. 平均预测置信度                  ← DA 应让 spot 预测更确定
# ============================================================
from scipy.stats import entropy
from topact import TopACT

if not ablation_results:
    print('⚠  ablation_results 为空，跳过 Stage 9.2')
    print('   请先在 Stage 0 设 RUN_ABLATION=True 并跑 Stage 9.1')
else:
    # 参考分布（scRNA 训练集真实分布）
    ref_dist      = np.bincount(flex_labels, minlength=n_classes) / len(flex_labels)
    ref_dist_safe = ref_dist + 1e-10

    xenium_metrics = {}
    for cfg_name, m in ablation_results.items():
        if 'spot_proba' not in m:
            print(f'  [{cfg_name}] 跳过（消融时未保存 spot_proba）')
            continue

        proba = m['spot_proba']
        pred  = proba.argmax(axis=1)

        # 1. KL(pred || ref)
        pred_dist = np.bincount(pred, minlength=n_classes) / len(pred) + 1e-10
        kl = float(entropy(pred_dist, ref_dist_safe))

        # 2. Moran's I（耗时 ~30s/配置）
        moran = TopACT.morans_i(labels=pred, coords=spot_coords, n_neighbors=10)

        # 3. 平均置信度
        mean_conf = float(proba.max(axis=1).mean())

        xenium_metrics[cfg_name] = {
            'kl_to_scrna': kl,
            'morans_I'   : moran['I'],
            'mean_conf'  : mean_conf,
            'val_f1'     : m['f1_macro'],
        }
        print(f"  [{cfg_name:<8}]  KL={kl:.3f}  Moran={moran['I']:.4f}  "
              f"Conf={mean_conf:.3f}  scRNA-F1={m['f1_macro']:.4f}")

    with open(ABL_XENIUM_FILE, 'wb') as f:
        pickle.dump(xenium_metrics, f)
    print(f"\n✅ Xenium 端消融指标已保存: {ABL_XENIUM_FILE}")

    # 自动算各组件对 KL 的贡献（DA 的真实作用集中体现在 KL 上）
    if 'full' in xenium_metrics and 'no_da' in xenium_metrics:
        delta_kl = xenium_metrics['no_da']['kl_to_scrna'] - \
                   xenium_metrics['full']['kl_to_scrna']
        print(f"\n说明: full vs no_da 的 KL 差 = {delta_kl:+.3f}")
        print(f"      （KL 越小 = 类分布越接近 scRNA 参考；DA 应使 KL 减小）")


## Stage 10 — 论文最终交付物

读取前面所有阶段的输出，生成毕业论文需要的：

- **`results/evaluation/thesis_tables.md`**：表 5-1（主对比）/ 5-2（消融）/ 5-5（Moran's I）
- **图 5-1**：方法对比柱状图
- **图 5-2**：最优 GNN 模型混淆矩阵
- **图 5-3**：消融训练曲线
- **图 5-5**：各细胞类型 per-class Moran's I
- **图 5-7**：UMAP 嵌入（按数据来源 + 按预测类型）
- **图 5-8**：spot 预测置信度分布（小提琴图 + CDF）

每张图都防御式地处理"上游 cell 没跑/数据缺失"的情况，不会因为单个 stage 没跑就崩。


In [ ]:
# ============================================================
# Stage 10.1 — 表 5-1 / 5-2 / 5-5 → markdown
# 数据源：
#   topact_metrics.pkl       (TopACT)
#   ablation_results.pkl     (消融，含 'full' 作为最优 GNN 主对比)
#   morans_all_methods.pkl   (Moran's I)
#   ablation_xenium_metrics.pkl (Xenium 端指标)
# ============================================================
OUT_MD = os.path.join(PATHS['output']['evaluation'], 'thesis_tables.md')
md = []

def fmt(x, fmt_str='.4f'):
    return f"{x:{fmt_str}}" if (x is not None and x == x) else "—"

# ── 表 5-1：主对比 ────────────────────────────────────────────
md.append('## 表 5-1  各方法在 scRNA 验证集上的定量性能对比\n')
md.append('| 方法 | 准确率 | F1-Macro | F1-Weighted | Kappa |')
md.append('| --- | --- | --- | --- | --- |')

# TopACT
if os.path.exists(TOPACT_METRIC_FILE):
    with open(TOPACT_METRIC_FILE, 'rb') as f: t = pickle.load(f)
    md.append(f"| TopACT (SVM 基线) | {fmt(t['acc'])} | {fmt(t['f1_macro'])} | "
              f"{fmt(t['f1_weighted'])} | {fmt(t['kappa'])} |")
else:
    md.append("| TopACT (SVM 基线) | — | — | — | — |  *(未运行 Stage 4.2)*")

# 各 GNN 模型（直接用 val_preds 实时算，最准确）
from sklearn.metrics import accuracy_score, f1_score, cohen_kappa_score
y_val_int = flex_labels[split_info['val_idx']]
for name in ['GCN', 'GraphSAGE', 'GAT']:
    if name in val_preds:
        vp = val_preds[name]
        acc = accuracy_score(y_val_int, vp)
        f1m = f1_score(y_val_int, vp, average='macro')
        f1w = f1_score(y_val_int, vp, average='weighted')
        kap = cohen_kappa_score(y_val_int, vp)
        star = ' ★' if name == best_name else ''
        md.append(f"| {name}{star} | {fmt(acc)} | {fmt(f1m)} | "
                  f"{fmt(f1w)} | {fmt(kap)} |")

md.append('')

# ── 表 5-2：消融 ──────────────────────────────────────────────
md.append(f'## 表 5-2  域适应组件消融实验（基础模型：GraphSAGE）\n')
md.append('| 配置 | λ_MMD | λ_Ent | λ_PL | 准确率 | F1-Macro | F1-Weighted | Kappa | KL→scRNA | Moran I |')
md.append('| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |')

label_map = {
    'full'  : '完整模型 (Full)',
    'no_pl' : '去掉伪标签 (w/o PL)',
    'no_ent': '去掉熵正则 (w/o Ent)',
    'no_mmd': '去掉 MMD (w/o MMD)',
    'no_da' : '去掉全部 DA (w/o DA)',
}

abl = ablation_results if ablation_results else {}
xen = {}
if os.path.exists(ABL_XENIUM_FILE):
    with open(ABL_XENIUM_FILE, 'rb') as f:
        xen = pickle.load(f)

for key in ['full', 'no_pl', 'no_ent', 'no_mmd', 'no_da']:
    if key not in abl:
        md.append(f"| {label_map[key]} | — | — | — | — | — | — | — | — | — |")
        continue
    m  = abl[key]
    xm = xen.get(key, {})
    md.append(
        f"| {label_map[key]} | "
        f"{m['lambda_mmd']:.2f} | {m['lambda_ent']:.2f} | {m['lambda_pl']:.2f} | "
        f"{fmt(m['accuracy'])} | {fmt(m['f1_macro'])} | "
        f"{fmt(m['f1_weighted'])} | {fmt(m['kappa'])} | "
        f"{fmt(xm.get('kl_to_scrna'), '.3f')} | "
        f"{fmt(xm.get('morans_I'), '.3f')} |"
    )

# 各组件对 F1 的贡献（pp）
if 'full' in abl and 'no_da' in abl:
    delta = (abl['full']['f1_macro'] - abl['no_da']['f1_macro']) * 100
    md.append(f"\n**说明**：完整 DA 相比 w/o DA 提升 F1-Macro **{delta:+.2f} pp**。")
    for key in ['no_pl', 'no_ent', 'no_mmd']:
        if key in abl:
            d = (abl['full']['f1_macro'] - abl[key]['f1_macro']) * 100
            comp = key.replace('no_', '').upper()
            md.append(f"- 去掉 {comp}：F1-Macro {d:+.2f} pp")
md.append('')

# ── 表 5-5：Moran's I ─────────────────────────────────────────
md.append("## 表 5-5  各方法预测结果的全局 Moran's I 空间自相关\n")
md.append("| 方法 | Moran's I | z-score | p-value |")
md.append("| --- | --- | --- | --- |")

if os.path.exists(MORAN_FILE):
    with open(MORAN_FILE, 'rb') as f: morans = pickle.load(f)
    for method in ['Random', 'TopACT', 'GCN', 'GraphSAGE', 'GAT']:
        if method not in morans: continue
        mr = morans[method]
        p_str = '< 0.001' if mr['p_value'] < 1e-3 else f"{mr['p_value']:.3e}"
        md.append(f"| {method} | {fmt(mr['I'])} | "
                  f"{fmt(mr['z_score'], '.2f')} | {p_str} |")
else:
    md.append("| — | — | — | — |  *(未运行 Stage 7)*")

# ── 写文件 ────────────────────────────────────────────────────
with open(OUT_MD, 'w', encoding='utf-8') as f:
    f.write('\n'.join(md))
print(f'✅ 论文表格已生成: {OUT_MD}')
print('\n' + '\n'.join(md))


In [ ]:
# ============================================================
# Stage 10.2 — 图 5-1 各方法定量对比柱状图
# ============================================================
import matplotlib.pyplot as plt
plt.rcParams['axes.unicode_minus'] = False

from sklearn.metrics import accuracy_score, f1_score, cohen_kappa_score

methods, acc_l, f1m_l, f1w_l, kap_l = [], [], [], [], []
y_val_int = flex_labels[split_info['val_idx']]

# TopACT（如果存在）
if os.path.exists(TOPACT_METRIC_FILE):
    with open(TOPACT_METRIC_FILE, 'rb') as f: t = pickle.load(f)
    methods.append('TopACT')
    acc_l.append(t['acc']);          f1m_l.append(t['f1_macro'])
    f1w_l.append(t['f1_weighted']);  kap_l.append(t['kappa'])

# 各 GNN
for name in ['GCN', 'GraphSAGE', 'GAT']:
    if name not in val_preds:
        continue
    vp   = val_preds[name]
    methods.append(f'{name}\n(★ 本文)' if name == best_name else name)
    acc_l.append(accuracy_score(y_val_int, vp))
    f1m_l.append(f1_score(y_val_int, vp, average='macro'))
    f1w_l.append(f1_score(y_val_int, vp, average='weighted'))
    kap_l.append(cohen_kappa_score(y_val_int, vp))

if not methods:
    print('⚠  无可用数据')
else:
    n = len(methods); x = np.arange(n); w = 0.2
    fig, ax = plt.subplots(figsize=(max(8, n*1.5), 5.5), dpi=200)
    b1 = ax.bar(x - 1.5*w, acc_l, w, label='Accuracy',    color='#4C72B0')
    b2 = ax.bar(x - 0.5*w, f1m_l, w, label='F1-Macro',    color='#DD8452')
    b3 = ax.bar(x + 0.5*w, f1w_l, w, label='F1-Weighted', color='#55A868')
    b4 = ax.bar(x + 1.5*w, kap_l, w, label='Kappa',       color='#C44E52')

    for bars in [b1, b2, b3, b4]:
        for b in bars:
            h = b.get_height()
            ax.text(b.get_x() + b.get_width()/2, h + 0.005,
                    f'{h:.3f}', ha='center', fontsize=8)

    ax.set_xticks(x); ax.set_xticklabels(methods)
    ax.set_ylabel('Score'); ax.set_ylim(0, 1.0)
    ax.set_title('各方法在 scRNA 验证集上的定量性能对比', fontsize=12)
    ax.legend(loc='lower right', frameon=True)
    ax.grid(axis='y', alpha=0.3)

    out = os.path.join(PATHS['figures'], 'fig_5_1_methods_comparison.png')
    plt.tight_layout(); plt.savefig(out, dpi=200, bbox_inches='tight')
    plt.show()
    print(f'✅ 图 5-1 已保存: {out}')


In [ ]:
# ============================================================
# Stage 10.3 — 图 5-2 最优 GNN 模型混淆矩阵（行归一化 = 召回率）
# ============================================================
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

print(f'最优模型: {best_name}')
y_val      = flex_labels[split_info['val_idx']]
y_pred     = val_preds[best_name]
labels_int = np.arange(n_classes)

cm     = confusion_matrix(y_val, y_pred, labels=labels_int)
cm_row = cm / cm.sum(axis=1, keepdims=True).clip(min=1)

fig, ax = plt.subplots(figsize=(11, 9), dpi=200)
im = ax.imshow(cm_row, cmap='Blues', vmin=0, vmax=1.0, aspect='auto')

short_labels = [c if len(c) <= 22 else c[:20] + '..' for c in cell_types_list]
ax.set_xticks(labels_int); ax.set_yticks(labels_int)
ax.set_xticklabels(short_labels, rotation=45, ha='right', fontsize=9)
ax.set_yticklabels(short_labels, fontsize=9)
ax.set_xlabel('Predicted', fontsize=11)
ax.set_ylabel('True',      fontsize=11)
ax.set_title(
    f'{best_name} 在 scRNA 验证集上的归一化混淆矩阵（行归一化=召回率）',
    fontsize=12,
)

for i in range(n_classes):
    for j in range(n_classes):
        v = cm_row[i, j]
        if v >= 0.01:
            ax.text(j, i, f'{v:.2f}', ha='center', va='center',
                    color='white' if v > 0.5 else 'black', fontsize=7)

cbar = plt.colorbar(im, ax=ax, fraction=0.045, pad=0.02)
cbar.set_label('Recall')

out = os.path.join(PATHS['figures'], 'fig_5_2_confusion_matrix.png')
plt.tight_layout(); plt.savefig(out, dpi=200, bbox_inches='tight')
plt.show()
print(f'✅ 图 5-2 已保存: {out}')

# 各类召回率排序（论文文本可引用）
print('\n各细胞类型召回率（从高到低）:')
for i in np.argsort(-np.diag(cm_row)):
    print(f'  {cell_types_list[i][:36]:<36}  {cm_row[i,i]:.3f}')


In [ ]:
# ============================================================
# Stage 10.4 — 图 5-3 消融实验训练曲线（仅当消融已跑）
# ============================================================
import matplotlib.pyplot as plt

if not ablation_results:
    print('⏭  消融实验未运行，跳过图 5-3')
else:
    fig, ax = plt.subplots(figsize=(9, 5), dpi=200)
    colors = {'full':'#C44E52', 'no_pl':'#DD8452', 'no_ent':'#55A868',
              'no_mmd':'#4C72B0', 'no_da':'#888888'}

    plotted = 0
    for name, m in ablation_results.items():
        h = m.get('history', [])
        if not h:
            continue
        # 兼容多种 history 字段名（不同训练代码的命名差异）
        first = h[0]
        f1_key = next((k for k in
                       ['val_f1_macro', 'val_f1', 'f1_macro', 'f1']
                       if k in first), None)
        if f1_key is None:
            print(f'  [{name}] history 缺少 F1 字段，可用键: {list(first.keys())}')
            continue
        f1_seq = [r[f1_key] for r in h]
        ax.plot(f1_seq, label=name, color=colors.get(name, '#000'),
                linewidth=1.5, alpha=0.8)
        plotted += 1

    if plotted == 0:
        print('⚠  所有消融配置的 history 都为空，跳过图 5-3')
    else:
        ax.set_xlabel('Epoch'); ax.set_ylabel('Val F1-Macro')
        ax.set_title('消融实验训练过程对比')
        ax.legend(); ax.grid(alpha=0.3)
        out = os.path.join(PATHS['figures'], 'fig_5_3_ablation_curves.png')
        plt.tight_layout(); plt.savefig(out, dpi=200, bbox_inches='tight')
        plt.show()
        print(f'✅ 图 5-3 已保存: {out}')


In [ ]:
# ============================================================
# Stage 10.5 — 图 5-5 各细胞类型 Moran's I（最优模型）
# ============================================================
import matplotlib.pyplot as plt
from topact import TopACT

best_pred = best_proba.argmax(axis=1)
print(f'计算 {best_name} 的 per-class Moran\'s I ...')
per_class = TopACT.per_class_morans_i(
    labels      = best_pred,
    coords      = spot_coords,
    cell_types  = cell_types_list,
    n_neighbors = 10,
)

# 排序（降序）
sorted_items = sorted(per_class.items(), key=lambda x: -x[1]['I'])
names_  = [k for k, _ in sorted_items]
values  = [v['I']       for _, v in sorted_items]
pvals   = [v['p_value'] for _, v in sorted_items]

fig, ax = plt.subplots(figsize=(11, 6), dpi=200)
short = [n if len(n) <= 22 else n[:20] + '..' for n in names_]
bar_colors = ['#2A6AB7' if v >= 0.5 else
              '#7DA9D6' if v >= 0.3 else
              '#BCC9D9' for v in values]

bars = ax.bar(range(len(names_)), values, color=bar_colors,
              edgecolor='#333', linewidth=0.4)

for b, v, p in zip(bars, values, pvals):
    star = '***' if p < 1e-3 else '**' if p < 1e-2 else '*' if p < 5e-2 else 'ns'
    ax.text(b.get_x() + b.get_width()/2, b.get_height() + 0.01,
            f'{v:.3f}\n{star}', ha='center', va='bottom', fontsize=8)

ax.axhline(0.5, color='gray',  linestyle='--', alpha=0.5,
           label='强空间自相关 (I=0.5)')
ax.axhline(0.3, color='gray',  linestyle=':',  alpha=0.5,
           label='中度空间自相关 (I=0.3)')
ax.axhline(0,   color='black', linewidth=0.5)

ax.set_xticks(range(len(names_)))
ax.set_xticklabels(short, rotation=40, ha='right', fontsize=9)
ax.set_ylabel("Moran's I", fontsize=11)
ax.set_ylim(min(0, min(values) * 1.1) - 0.05, max(values) * 1.2)
ax.set_title(f'{best_name} 预测结果各细胞类型 Moran\'s I 空间自相关',
             fontsize=12)
ax.legend(loc='upper right', fontsize=9)
ax.grid(axis='y', alpha=0.3)

ax.text(0.02, 0.97, '*** p<0.001  ** p<0.01  * p<0.05  ns 不显著',
        transform=ax.transAxes, fontsize=8, va='top',
        bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8))

out = os.path.join(PATHS['figures'], 'fig_5_5_per_class_morans.png')
plt.tight_layout(); plt.savefig(out, dpi=200, bbox_inches='tight')
plt.show()
print(f'✅ 图 5-5 已保存: {out}')


In [ ]:
# ============================================================
# Stage 10.6 — 图 5-7 隐层 UMAP 嵌入
#   (a) 按数据来源 (scRNA / Xenium)
#   (b) 按预测细胞类型
# 子采样：scRNA 全部 + spot 5000（30 万 spot 全画太密）
# ============================================================
import matplotlib.pyplot as plt

best_model = gnn_results.get(best_name, {}).get('model')
if best_model is None:
    print(f'⚠  {best_name} 的 model 对象不可用（可能从历史 summary 加载只有指标）')
    print('   请重训该模型或重跑 Stage 4.1 推断 cell。本次跳过图 5-7。')
else:
    try:
        import umap
    except ImportError:
        print('⚠  umap-learn 未安装，跳过图 5-7')
    else:
        # 隐层 forward
        best_model = best_model.cpu().eval()
        with torch.no_grad():
            h = best_model.encode(data.cpu()).numpy()
        print(f'隐层维度: {h.shape}')

        # 子采样
        np.random.seed(42)
        n_sc   = data.n_scrna
        sc_idx = np.arange(n_sc)
        n_sp_sample = min(5000, len(h) - n_sc)
        sp_idx  = np.random.choice(np.arange(n_sc, len(h)), n_sp_sample,
                                   replace=False)
        sub_idx = np.concatenate([sc_idx, sp_idx])
        h_sub   = h[sub_idx]

        print(f'UMAP 投影中（约 1-3 分钟，{len(h_sub):,} 点）...')
        emb = umap.UMAP(n_neighbors=30, min_dist=0.1,
                        random_state=42).fit_transform(h_sub)

        fig, axes = plt.subplots(1, 2, figsize=(14, 6), dpi=200)

        # (a) 按数据来源
        src = np.array(['scRNA']*len(sc_idx) + ['Xenium']*len(sp_idx))
        for s, c in [('scRNA', '#4C72B0'), ('Xenium', '#DD8452')]:
            mask = src == s
            axes[0].scatter(emb[mask, 0], emb[mask, 1], s=2, c=c,
                            alpha=0.5, label=s)
        axes[0].legend(); axes[0].set_title('(a) 按数据来源')
        axes[0].set_xlabel('UMAP-1'); axes[0].set_ylabel('UMAP-2')

        # (b) 按预测细胞类型
        all_pred = np.concatenate([flex_labels, best_proba.argmax(axis=1)])
        preds_sub = all_pred[sub_idx]
        axes[1].scatter(emb[:, 0], emb[:, 1], s=2, c=preds_sub,
                        cmap='tab20', alpha=0.6)
        axes[1].set_title('(b) 按预测细胞类型')
        axes[1].set_xlabel('UMAP-1'); axes[1].set_ylabel('UMAP-2')

        out = os.path.join(PATHS['figures'], 'fig_5_7_umap.png')
        plt.tight_layout(); plt.savefig(out, dpi=200, bbox_inches='tight')
        plt.show()
        print(f'✅ 图 5-7 已保存: {out}')


In [ ]:
# ============================================================
# Stage 10.7 — 图 5-8 spot 端预测置信度分布
#   (a) 各方法置信度小提琴图
#   (b) 累积分布函数 (CDF)
# ============================================================
import matplotlib.pyplot as plt

# 显示顺序：GNN → 基线
ordered = [n for n in spot_probas if is_gnn(n)] + \
          [n for n in spot_probas if not is_gnn(n)]

fig, axes = plt.subplots(1, 2, figsize=(13, 5), dpi=200)

# (a) 小提琴图
data_v   = []
labels_v = []
for name in ordered:
    conf = spot_probas[name].max(axis=1)
    data_v.append(conf)
    labels_v.append(name)

axes[0].violinplot(data_v, showmedians=True)
axes[0].set_xticks(range(1, len(labels_v) + 1))
axes[0].set_xticklabels(labels_v)
axes[0].set_ylabel('Max softmax probability')
axes[0].set_title('(a) 各方法 spot 预测置信度分布')
axes[0].axhline(0.9, color='r', linestyle='--', alpha=0.5,
                label='伪标签阈值 0.9')
axes[0].legend(); axes[0].grid(alpha=0.3)

# (b) CDF
for name in ordered:
    conf = np.sort(spot_probas[name].max(axis=1))
    cdf  = np.arange(len(conf)) / len(conf)
    axes[1].plot(conf, cdf, label=name, linewidth=1.5)
axes[1].axvline(0.9, color='r', linestyle='--', alpha=0.5)
axes[1].set_xlabel('Confidence')
axes[1].set_ylabel('Cumulative fraction')
axes[1].set_title('(b) 置信度累积分布')
axes[1].legend(); axes[1].grid(alpha=0.3)

out = os.path.join(PATHS['figures'], 'fig_5_8_confidence.png')
plt.tight_layout(); plt.savefig(out, dpi=200, bbox_inches='tight')
plt.show()
print(f'✅ 图 5-8 已保存: {out}')

# 文本统计（论文可引用）
print('\n各方法高置信度（>0.9）spot 占比:')
for name in ordered:
    conf = spot_probas[name].max(axis=1)
    pct  = (conf >= 0.9).mean() * 100
    tag  = 'GNN' if is_gnn(name) else 'baseline'
    print(f'  [{tag:<8}] {name:<12}  {pct:>5.1f}%')


## 流程结束 ✓

执行完所有 stage 后，论文需要的产出位置如下：

### 表格（直接复制到论文）
- `results/evaluation/thesis_tables.md` — 表 5-1 / 5-2 / 5-5 的 markdown

### 图（已自动保存到 `figures/`）
- `fig_3_1_transcript_density.png` — 图 3-1 转录本密度
- `spot_spatial_all_models.png` — 各方法空间预测对比
- `spatial_comparison_gnn_vs_seurat.png` — 与 Seurat 并排对比
- `proportion_comparison.png` — 类型比例堆叠条形
- `fig_5_1_methods_comparison.png` — 图 5-1 方法对比柱状
- `fig_5_2_confusion_matrix.png` — 图 5-2 混淆矩阵
- `fig_5_3_ablation_curves.png` — 图 5-3 消融训练曲线
- `fig_5_5_per_class_morans.png` — 图 5-5 各类 Moran's I
- `fig_5_7_umap.png` — 图 5-7 UMAP 嵌入
- `fig_5_8_confidence.png` — 图 5-8 置信度分布

### 还需要 draw.io / PowerPoint 手画的（无法自动生成）
- 图 3-2  bin 聚合示意图
- 图 4-1  整体框架图
- 图 4-2  联合图结构示意

### 缓存数据（中间产物，可保留）
- `data/cache/export/`  — scRNA numpy 缓存
- `data/cache/graph/`   — spot 图、PCA、转录本下采样
- `results/models/`     — GNN 权重 + 训练 summary
- `results/predictions/` — spot 概率 + CSV 预测
- `results/evaluation/` — 各种 pkl + thesis_tables.md
